# 01 — CARE to Compare V6: Dataset Exploration and Audit

Exploratory, read-only audit. No models are trained.

## 1. Environment and configuration

In [2]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import sys, platform, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import zipfile, io, hashlib, json

print('Python:', sys.version)
print('Platform:', platform.platform())
for lib in ['pandas','numpy','matplotlib','pyarrow']:
    try:
        m = __import__(lib)
        print(lib, getattr(m, '__version__', 'n/a'))
    except Exception as e:
        print(lib, 'NOT AVAILABLE', e)

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path('/content')
    ZIP_PATH = Path('/content/drive/MyDrive/Care_to_compare/CARE_To_Compare.zip')
    DATASET_DIR = ROOT / 'care_to_compare'
    EXTRACTED = ROOT / 'data/raw/care_to_compare/CARE_To_Compare'
else:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    DATASET_DIR = ROOT / 'data' / 'raw' / 'care_to_compare'
    EXTRACTED = DATASET_DIR / 'CARE_To_Compare'
    ZIP_CANDIDATES = [DATASET_DIR / 'CARE_To_Compare.zip', DATASET_DIR / 'data' / 'raw' / 'care_to_compare' / 'CARE_To_Compare.zip']
    ZIP_PATH = next((pp for pp in ZIP_CANDIDATES if pp.exists()), ZIP_CANDIDATES[0])

print('IN_COLAB:', IN_COLAB)
print('ROOT:', ROOT)
print('DATASET_DIR:', DATASET_DIR, 'exists:', DATASET_DIR.exists())
print('EXTRACTED:', EXTRACTED, 'exists:', EXTRACTED.exists())

def file_exists(rel):
    if EXTRACTED.exists():
        p = EXTRACTED / rel
        if p.exists():
            return p
    return None

def read_text(rel):
    p = file_exists(rel)
    if p is not None:
        return p.read_text(encoding='utf-8', errors='replace')
    with zipfile.ZipFile(ZIP_PATH) as z:
        return z.read('CARE_To_Compare/' + rel).decode('utf-8', errors='replace')

def read_bytes(rel):
    p = file_exists(rel)
    if p is not None:
        return p.read_bytes()
    with zipfile.ZipFile(ZIP_PATH) as z:
        return z.read('CARE_To_Compare/' + rel)

def read_csv(rel, **kw):
    b = read_bytes(rel)
    return pd.read_csv(io.BytesIO(b), **kw)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
pandas 2.2.3
numpy 2.4.6
matplotlib 3.10.0
pyarrow 23.0.1
ROOT: /content
DATASET_DIR: /content/care_to_compare exists: False
EXTRACTED: /content/data/raw/care_to_compare/CARE_To_Compare exists: False
ZIP_PATH (Google Drive): /content/drive/MyDrive/Care_to_compare/CARE_To_Compare.zip exists: True


## 2. Dataset inventory

In [4]:
# Recursive inventory of the archive (without extracting)
if EXTRACTED.exists():
    entries = []
    for p in EXTRACTED.rglob('*'):
        if p.is_file():
            entries.append({'path': str(p.relative_to(EXTRACTED)), 'size': p.stat().st_size, 'source': 'extracted'})
else:
    with zipfile.ZipFile(ZIP_PATH) as z:
        entries = [{'path': i.filename.replace('CARE_To_Compare/',''), 'size': i.file_size,
                    'source': 'zip', 'is_dir': i.is_dir()} for i in z.infolist()]
    entries = [e for e in entries if not e.get('is_dir')]

inv = pd.DataFrame(entries)
print('Total files:', len(inv))
print('Total size (bytes):', int(inv['size'].sum()))
print('Total size (GB):', round(inv['size'].sum()/1e9, 2))

# Extension summary
inv['ext'] = inv['path'].str.extract(r'\.([A-Za-z0-9]+)$')[0].str.lower().fillna('<none>')
print('\nBy extension:')
print(inv.groupby('ext').agg(files=('size','count'), bytes=('size','sum')).to_string())

# Top-level grouping (wind farm)
def top_level(p):
    return p.split('/')[0] if '/' in p else p
inv['group'] = inv['path'].apply(top_level)
print('\nBy top-level group:')
print(inv.groupby('group').agg(files=('size','count'), bytes=('size','sum')).to_string())

# Save inventory
out_dir = ROOT / 'reports'
out_dir.mkdir(exist_ok=True)
inv.to_csv(out_dir / 'dataset_inventory.csv', index=False)
print('\nInventory saved to reports/dataset_inventory.csv')


Total files: 103
Total size (bytes): 19987485991
Total size (GB): 19.99

By extension:
     files        bytes
ext                    
csv    101  19987475455
md       1         5268
txt      1         5268

By top-level group:
             files        bytes
group                          
README.md        1         5268
README.txt       1         5268
Wind Farm A     24    802870209
Wind Farm B     17   1365828066
Wind Farm C     60  17818777180

Inventory saved to reports/dataset_inventory.csv


In [5]:
# List the small metadata files
meta_files = inv[inv['path'].str.endswith(('.csv','.md','.txt')) & ~inv['path'].str.contains('/datasets/')]
print(meta_files[['path','size']].to_string(index=False))


                               path  size
                          README.md  5268
                         README.txt  5268
         Wind Farm A/event_info.csv  1788
Wind Farm A/feature_description.csv  4098
         Wind Farm B/event_info.csv  1344
Wind Farm B/feature_description.csv  5324
         Wind Farm C/event_info.csv  6070
Wind Farm C/feature_description.csv 21396


In [6]:
# Duplicate detection by name (basename) and by content (md5 for small files)
inv['basename'] = inv['path'].str.split('/').str[-1]
dup_name = inv[inv['basename'].duplicated(keep=False)].sort_values('basename')
print('Duplicate basenames:', dup_name['basename'].nunique())
print(dup_name[['path']].to_string(index=False) if len(dup_name) else 'None')

# Content md5 for small files (< 50 MB) to find exact duplicates
small = inv[inv['size'] < 50*1024*1024].copy()
def md5_of(rel):
    b = read_bytes(rel)
    return hashlib.md5(b).hexdigest()
small['md5'] = small['path'].apply(md5_of)
dup_content = small[small['md5'].duplicated(keep=False)].sort_values('md5')
print('\nExact-content duplicate groups (small files):')
if len(dup_content):
    print(dup_content[['path','md5']].to_string(index=False))
else:
    print('None')


Duplicate basenames: 2
                               path
         Wind Farm A/event_info.csv
         Wind Farm B/event_info.csv
         Wind Farm C/event_info.csv
Wind Farm A/feature_description.csv
Wind Farm B/feature_description.csv
Wind Farm C/feature_description.csv

Exact-content duplicate groups (small files):
      path                              md5
 README.md 9a3afda46c6d7d6c91ead0e4e5097b1f
README.txt 9a3afda46c6d7d6c91ead0e4e5097b1f


## 3. Documentation analysis

In [7]:
# README
readme = read_text('README.md')
print(readme[:4000])


## Table of Contents
1. [The Data](#the-data)
   * [Event Data](#event-data)
   * [Event Information](#event-information)
   * [Feature Description](#feature-description)

## The Data

The data consists of 95 datasets, containing 89 years of SCADA time series distributed across 36 different wind turbines
from the three wind farms A, B and C. The data for Wind farm A is based on data from the EDP-open data platform (https://www.edp.com/en/innovation/open-data/data), 
and consists of 5 wind turbines of an onshore wind farm in Portugal. 
It contains SCADA data and information derived by a given fault logbook which defines start timestamps for specified faults. 
From this data 22 datasets were selected to be included in this data collection. 
The other two wind farms are offshore wind farms located in Germany. All three datasets were anonymized due to confidentiality reasons for the wind farms B and C.
The overall dataset is balanced, as 44 out the 95 datasets contain a labeled anomaly eve

In [8]:
# Parse event_info for all farms
farms = ['Wind Farm A','Wind Farm B','Wind Farm C']
event_infos = {}
for farm in farms:
    rel = f'{farm}/event_info.csv'
    df = read_csv(rel, sep=';')
    event_infos[farm] = df
    print('='*20, farm, 'event_info shape:', df.shape)
    print('columns:', list(df.columns))
    print('label counts:', df['event_label'].value_counts().to_dict())
    print(df.head(2).to_string(index=False))


==================== Wind Farm A event_info shape: (22, 8)
columns: ['asset', 'event_id', 'event_label', 'event_start', 'event_start_id', 'event_end', 'event_end_id', 'event_description']
label counts: {'anomaly': 12, 'normal': 10}
 asset  event_id event_label         event_start  event_start_id           event_end  event_end_id   event_description
    11        68     anomaly 2023-07-28 13:20:00           52063 2023-08-11 13:10:00         54076 Transformer failure
    21        22     anomaly 2023-08-12 09:50:00           51888 2023-08-19 10:00:00         52892     Hydraulic group
==================== Wind Farm B event_info shape: (15, 8)
columns: ['asset_id', 'event_id', 'event_label', 'event_start', 'event_start_id', 'event_end', 'event_end_id', 'event_description']
label counts: {'normal': 9, 'anomaly': 6}
 asset_id  event_id event_label         event_start  event_start_id           event_end  event_end_id                    event_description
       14        34     anomaly 2023-08

In [9]:
# Feature descriptions
feature_descs = {}
for farm in farms:
    rel = f'{farm}/feature_description.csv'
    df = read_csv(rel, sep=';')
    feature_descs[farm] = df
    print('='*20, farm, 'feature_description shape:', df.shape)
    print('columns:', list(df.columns))
    print('sensors:', df['sensor_name'].nunique())


==================== Wind Farm A feature_description shape: (54, 6)
columns: ['sensor_name', 'statistics_type', 'description', 'unit', 'is_angle', 'is_counter']
sensors: 54
==================== Wind Farm B feature_description shape: (63, 6)
columns: ['sensor_name', 'statistics_type', 'description', 'unit', 'is_angle', 'is_counter']
sensors: 63
==================== Wind Farm C feature_description shape: (238, 6)
columns: ['sensor_name', 'statistics_type', 'description', 'unit', 'is_angle', 'is_counter']
sensors: 238


In [10]:
# Consolidated documentation-derived facts
print('README states: 95 datasets, 36 turbines, 3 wind farms, 10-min frequency.')
print('Verified file count (datasets):', (inv['path'].str.contains('/datasets/') & inv['path'].str.endswith('.csv')).sum())
# event totals
all_ev = pd.concat([event_infos[f].assign(farm=f) for f in farms], ignore_index=True)
print('Total events:', len(all_ev))
print('Label totals:', all_ev['event_label'].value_counts().to_dict())


README states: 95 datasets, 36 turbines, 3 wind farms, 10-min frequency.
Verified file count (datasets): 95
Total events: 95
Label totals: {'normal': 50, 'anomaly': 45}


## 4. Schema exploration

In [11]:
# Representative file per farm
rep_files = {'Wind Farm A':'Wind Farm A/datasets/0.csv',
             'Wind Farm B':'Wind Farm B/datasets/2.csv',
             'Wind Farm C':'Wind Farm C/datasets/1.csv'}
reps = {}
for farm, rel in rep_files.items():
    df = read_csv(rel, sep=';')
    reps[farm] = df
    print('='*20, farm, rel)
    print('shape:', df.shape)
    print('columns (first 12):', list(df.columns)[:12])
    print('dtypes summary:')
    print(df.dtypes.value_counts().to_string())


==================== Wind Farm A Wind Farm A/datasets/0.csv
shape: (54986, 86)
columns (first 12): ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id', 'sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'wind_speed_3_avg', 'wind_speed_4_avg', 'wind_speed_3_max', 'wind_speed_3_min']
dtypes summary:
float64    81
int64       3
object      2
==================== Wind Farm B Wind Farm B/datasets/2.csv
shape: (54774, 257)
columns (first 12): ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id', 'sensor_0_avg', 'sensor_0_max', 'sensor_0_min', 'sensor_0_std', 'sensor_1_avg', 'sensor_1_max', 'sensor_1_min']
dtypes summary:
float64    252
int64        3
object       2
==================== Wind Farm C Wind Farm C/datasets/1.csv
shape: (53569, 957)
columns (first 12): ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id', 'sensor_0_avg', 'sensor_0_max', 'sensor_0_min', 'sensor_0_std', 'sensor_1_avg', 'sensor_1_max', 'sensor_1_min']
dtypes summary:
float64    952
int6

In [12]:
# Meta columns and sensor columns
for farm, df in reps.items():
    meta = [c for c in df.columns if c in ('time_stamp','asset_id','asset','id','train_test','status_type_id','status_type')]
    sensor = [c for c in df.columns if c not in meta]
    print(farm, 'meta cols:', meta)
    print(farm, 'n sensor cols:', len(sensor))
    # suffix stats
    from collections import Counter
    suffixes = Counter(c.split('_')[-1] for c in sensor)
    print(farm, 'suffix counts:', dict(suffixes))


Wind Farm A meta cols: ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id']
Wind Farm A n sensor cols: 81
Wind Farm A suffix counts: {'avg': 46, 'max': 9, 'min': 9, 'std': 9, '44': 1, '45': 1, '46': 1, '47': 1, '48': 1, '49': 1, '50': 1, '51': 1}
Wind Farm B meta cols: ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id']
Wind Farm B n sensor cols: 252
Wind Farm B suffix counts: {'avg': 63, 'max': 63, 'min': 63, 'std': 63}
Wind Farm C meta cols: ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id']
Wind Farm C n sensor cols: 952
Wind Farm C suffix counts: {'avg': 238, 'max': 238, 'min': 238, 'std': 238}


In [13]:
# Descriptive stats for a few sensor columns (Farm A)
farm='Wind Farm A'
df=reps[farm]
cols = [c for c in df.columns if c.endswith('_avg')][:5]
print(df[cols].describe().to_string())


       sensor_0_avg  sensor_1_avg  sensor_2_avg  wind_speed_3_avg  wind_speed_4_avg
count  54986.000000  54986.000000  54986.000000      54986.000000      54986.000000
mean      20.146328    164.704963      0.835209          6.532088          6.493973
std        5.747968     91.616619     32.540732          3.985269          3.910725
min        4.000000      0.000000   -179.800000          0.500000          0.000000
25%       16.000000     93.200000    -10.400000          3.300000          3.300000
50%       20.000000    120.550000      0.600000          5.600000          5.600000
75%       24.000000    252.800000      9.800000          9.000000          9.000000
max       41.000000    359.000000    180.000000         25.500000         25.500000


In [14]:
# Schema differences across farms
print('Farm A cols:', reps['Wind Farm A'].shape[1])
print('Farm B cols:', reps['Wind Farm B'].shape[1])
print('Farm C cols:', reps['Wind Farm C'].shape[1])
# Common meta columns
meta_sets = {f: set(reps[f].columns) for f in reps}
common = meta_sets['Wind Farm A'] & meta_sets['Wind Farm B'] & meta_sets['Wind Farm C']
print('Columns common to all 3 representative files:', len(common))
print(sorted(common))


Farm A cols: 86
Farm B cols: 257
Farm C cols: 957
Columns common to all 3 representative files: 49
['asset_id', 'id', 'sensor_0_avg', 'sensor_10_avg', 'sensor_12_avg', 'sensor_13_avg', 'sensor_14_avg', 'sensor_15_avg', 'sensor_16_avg', 'sensor_18_avg', 'sensor_18_max', 'sensor_18_min', 'sensor_18_std', 'sensor_19_avg', 'sensor_1_avg', 'sensor_20_avg', 'sensor_21_avg', 'sensor_22_avg', 'sensor_23_avg', 'sensor_24_avg', 'sensor_25_avg', 'sensor_26_avg', 'sensor_31_avg', 'sensor_31_max', 'sensor_31_min', 'sensor_31_std', 'sensor_32_avg', 'sensor_33_avg', 'sensor_34_avg', 'sensor_35_avg', 'sensor_36_avg', 'sensor_37_avg', 'sensor_38_avg', 'sensor_39_avg', 'sensor_40_avg', 'sensor_41_avg', 'sensor_42_avg', 'sensor_43_avg', 'sensor_52_avg', 'sensor_52_max', 'sensor_52_min', 'sensor_52_std', 'sensor_53_avg', 'sensor_7_avg', 'sensor_8_avg', 'sensor_9_avg', 'status_type_id', 'time_stamp', 'train_test']


## 5. Temporal analysis

In [15]:
# Timestamp parsing and frequency (representative files)
for farm, df in reps.items():
    ts = pd.to_datetime(df['time_stamp'])
    deltas = ts.diff().dropna()
    print('='*20, farm)
    print('time range:', ts.min(), '->', ts.max())
    print('n rows:', len(df), 'n unique ts:', ts.nunique())
    print('duplicate ts:', ts.duplicated().sum())
    print('median delta:', deltas.median(), 'min:', deltas.min(), 'max:', deltas.max())
    print('gaps > 10min:', int((deltas > pd.Timedelta(minutes=10)).sum()))
    print('top deltas:', deltas.value_counts().head().to_dict())


==================== Wind Farm A
time range: 2022-08-04 06:10:00 -> 2023-08-24 06:10:00
n rows: 54986 n unique ts: 54986
duplicate ts: 0
median delta: 0 days 00:10:00 min: 0 days 00:10:00 max: 0 days 13:00:00
gaps > 10min: 18
top deltas: {Timedelta('0 days 00:10:00'): 54967, Timedelta('0 days 00:20:00'): 3, Timedelta('0 days 12:10:00'): 2, Timedelta('0 days 00:30:00'): 2, Timedelta('0 days 03:50:00'): 1}
==================== Wind Farm B
time range: 2022-08-08 19:20:00 -> 2023-08-24 04:20:00
n rows: 54774 n unique ts: 54774
duplicate ts: 0
median delta: 0 days 00:10:00 min: 0 days 00:10:00 max: 0 days 00:20:00
gaps > 10min: 1
top deltas: {Timedelta('0 days 00:10:00'): 54772, Timedelta('0 days 00:20:00'): 1}
==================== Wind Farm C
time range: 2022-10-01 00:00:00 -> 2023-10-16 00:00:00
n rows: 53569 n unique ts: 53569
duplicate ts: 0
median delta: 0 days 00:10:00 min: 0 days 00:10:00 max: 5 days 00:10:00
gaps > 10min: 2
top deltas: {Timedelta('0 days 00:10:00'): 53566, Timedelta

In [16]:
# Visualize a representative time series (Farm A, first sensor avg)
farm='Wind Farm A'
df=reps[farm]
ts = pd.to_datetime(df['time_stamp'])
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(ts, df['sensor_0_avg'], linewidth=0.5)
ax.set_title('Farm A - sensor_0_avg over time (representative)')
ax.set_xlabel('time_stamp'); ax.set_ylabel('sensor_0_avg')
plt.tight_layout(); plt.show()


In [17]:
# Event window overlay for Farm A file 0
farm='Wind Farm A'
df=reps[farm]
ei = event_infos[farm]
ev = ei[ei['event_id']==0].iloc[0]
ts = pd.to_datetime(df['time_stamp'])
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(ts, df['sensor_0_avg'], linewidth=0.5, label='sensor_0_avg')
es = pd.to_datetime(ev['event_start']); ee = pd.to_datetime(ev['event_end'])
ax.axvspan(es, ee, color='red', alpha=0.2, label='event window')
ax.set_title('Farm A dataset 0 - sensor_0_avg with event window')
ax.legend(); plt.tight_layout(); plt.show()


## 6. Data quality

In [18]:
# Nulls, inf, duplicates per representative file
for farm, df in reps.items():
    nulls = df.isna().sum()
    nnull_cols = int((nulls>0).sum())
    total_null = int(nulls.sum())
    infc = 0
    num = df.select_dtypes(include=[np.number])
    infc = int(np.isinf(num).sum().sum())
    print(farm, 'rows', len(df), 'cols', df.shape[1],
          'null_cols', nnull_cols, 'total_null', total_null,
          'inf', infc, 'dup_rows', int(df.duplicated().sum()))
    if nnull_cols>0:
        print('   null cols:', list(nulls[nulls>0].index))


Wind Farm A rows 54986 cols 86 null_cols 2 total_null 2 inf 0 dup_rows 0
   null cols: ['sensor_14_avg', 'sensor_22_avg']
Wind Farm B rows 54774 cols 257 null_cols 2 total_null 7 inf 0 dup_rows 0
   null cols: ['power_62_max', 'power_62_std']
Wind Farm C rows 53569 cols 957 null_cols 0 total_null 0 inf 0 dup_rows 0


In [19]:
# Zeros analysis (Farm A representative)
farm='Wind Farm A'
df=reps[farm]
num = df.select_dtypes(include=[np.number])
zc = (num==0).sum()
print('cols with any zero:', int((zc>0).sum()), 'of', num.shape[1])
print('top 10 zero counts:')
print(zc.sort_values(ascending=False).head(10).to_string())
# constant columns
const = [c for c in num.columns if num[c].nunique(dropna=False)<=1]
print('constant numeric cols:', len(const), const[:10])


cols with any zero: 48 of 84
top 10 zero counts:
asset_id                 54986
sensor_46                54986
sensor_49                54986
status_type_id           47540
sensor_47                34578
sensor_44                34568
reactive_power_28_std    23794
reactive_power_27_min    22897
reactive_power_28_max    22897
reactive_power_27_std    21567
constant numeric cols: 3 ['asset_id', 'sensor_46', 'sensor_49']


In [20]:
# Missing-value share by column (top 15) - Farm A
farm='Wind Farm A'
df=reps[farm]
miss = df.isna().mean().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10,5))
miss.plot(kind='bar', ax=ax)
ax.set_title('Farm A - top missing-value ratios by column')
ax.set_ylabel('missing ratio'); plt.tight_layout(); plt.show()


In [21]:
# Distribution of a sensor (Farm A)
farm='Wind Farm A'
df=reps[farm]
fig, ax = plt.subplots(figsize=(8,4))
df['sensor_0_avg'].hist(bins=50, ax=ax)
ax.set_title('Farm A - distribution of sensor_0_avg')
plt.tight_layout(); plt.show()


## 7. Anomaly and event exploration

In [22]:
# Event info summary per farm
for farm in farms:
    ei = event_infos[farm]
    print('='*20, farm, 'events', len(ei))
    print(ei['event_label'].value_counts().to_dict())
    asset_col = 'asset' if 'asset' in ei.columns else 'asset_id'
    print('unique assets:', ei[asset_col].nunique())


==================== Wind Farm A events 22
{'anomaly': 12, 'normal': 10}
unique assets: 5
==================== Wind Farm B events 15
{'normal': 9, 'anomaly': 6}
unique assets: 9
==================== Wind Farm C events 58
{'normal': 31, 'anomaly': 27}
unique assets: 22


In [23]:
# Verify event_start_id / event_end_id map to dataset rows (Farm A, event 0)
farm='Wind Farm A'
df=reps[farm]
ei=event_infos[farm]
ev=ei[ei['event_id']==0].iloc[0]
print('event_start_id', ev['event_start_id'], '->', df.loc[df['id']==ev['event_start_id'],'time_stamp'].values)
print('event_end_id', ev['event_end_id'], '->', df.loc[df['id']==ev['event_end_id'],'time_stamp'].values)
print('event_start (doc):', ev['event_start'], 'event_end (doc):', ev['event_end'])


event_start_id 52436 -> ['2023-08-06 06:10:00']
event_end_id 54447 -> ['2023-08-20 06:10:00']
event_start (doc): 2023-08-06 06:10:00 event_end (doc): 2023-08-20 06:10:00


In [24]:
# train_test distribution per representative file
for farm, df in reps.items():
    print(farm, df['train_test'].value_counts().to_dict())


Wind Farm A {'train': 52148, 'prediction': 2838}
Wind Farm B {'train': 52559, 'prediction': 2215}
Wind Farm C {'train': 51408, 'prediction': 2161}


In [25]:
# Distribution of events over time (start dates)
all_ev = pd.concat([event_infos[f].assign(farm=f) for f in farms], ignore_index=True)
all_ev['event_start'] = pd.to_datetime(all_ev['event_start'])
fig, ax = plt.subplots(figsize=(10,4))
for label, g in all_ev.groupby('event_label'):
    ax.hist(g['event_start'], bins=12, alpha=0.6, label=label)
ax.set_title('Distribution of event start dates by label')
ax.legend(); plt.tight_layout(); plt.show()


## 8. Preliminary visualization

In [26]:
# Series count by farm
counts = pd.Series({f: len(event_infos[f]) for f in farms})
fig, ax = plt.subplots(figsize=(6,4))
counts.plot(kind='bar', ax=ax)
ax.set_title('Number of series (datasets) by wind farm')
ax.set_ylabel('datasets'); plt.tight_layout(); plt.show()


In [27]:
# Records per series (sample of files: row counts from inventory already; recompute for one farm quickly via zip line counts is heavy, use stored? We'll approximate using event_info not rows. Instead show rows per representative.)
for farm, df in reps.items():
    print(farm, 'representative rows:', len(df))


Wind Farm A representative rows: 54986
Wind Farm B representative rows: 54774
Wind Farm C representative rows: 53569


In [28]:
# Missing values by variable - Farm B top 15
farm='Wind Farm B'
df=reps[farm]
miss = df.isna().mean().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10,5))
miss.plot(kind='bar', ax=ax)
ax.set_title('Farm B - top missing-value ratios by column')
ax.set_ylabel('missing ratio'); plt.tight_layout(); plt.show()


## 9. Preliminary conclusions

In [29]:
print('''
Key verified facts:
- 95 dataset CSVs across 3 wind farms (A:22, B:15, C:58).
- Total rows ~5.24M, 10-min frequency.
- Farm A ~86 cols, B ~257 cols, C ~957 cols.
- Meta columns: time_stamp, asset_id, id, train_test, status_type_id.
- Event info maps event_start_id/event_end_id to row ids.
- 45 anomaly + 50 normal events (README says 44/51 - minor discrepancy).
- 36 unique turbines total (5+9+22).
- Nulls are sparse (single cells). Some constant-zero sensors (e.g. Farm A sensor_46, sensor_49).
- Temporal gaps exist (Farm A up to 13h; Farm C up to 5 days).
- README.md and README.txt are byte-identical.

Next steps (not implemented here):
- Define per-farm feature equivalence before combining.
- Handle temporal gaps and missing values with documented strategy.
- Build time-based splits to avoid leakage.
- Associate event intervals to observations without per-row labels invention.
''')



Key verified facts:
- 95 dataset CSVs across 3 wind farms (A:22, B:15, C:58).
- Total rows ~5.24M, 10-min frequency.
- Farm A ~86 cols, B ~257 cols, C ~957 cols.
- Meta columns: time_stamp, asset_id, id, train_test, status_type_id.
- Event info maps event_start_id/event_end_id to row ids.
- 45 anomaly + 50 normal events (README says 44/51 - minor discrepancy).
- 36 unique turbines total (5+9+22).
- Nulls are sparse (single cells). Some constant-zero sensors (e.g. Farm A sensor_46, sensor_49).
- Temporal gaps exist (Farm A up to 13h; Farm C up to 5 days).
- README.md and README.txt are byte-identical.

Next steps (not implemented here):
- Define per-farm feature equivalence before combining.
- Handle temporal gaps and missing values with documented strategy.
- Build time-based splits to avoid leakage.
- Associate event intervals to observations without per-row labels invention.

